# cuDF Feature Engineering with GPU Checkpointing

A realistic feature engineering pipeline using cudf with GPU-side checkpointing
enabled throughout. All checkpoint and diff operations stay on the GPU.

In [ ]:
import cudf
import numpy as np

In [ ]:
%cudf_gpu_checkpoint on

## Load Data

Build a large DataFrame with mixed dtypes (4M rows, ~300 columns).

In [ ]:
n_rows = 4_000_000
rng = np.random.default_rng(42)
data = {}

# 21 factorized categoricals (float32)
for i in range(21):
    data[f'cat_{i}'] = rng.integers(0, 50, size=n_rows).astype(np.float32)

# Numeric targets (float64)
data['weight'] = rng.normal(50, 10, size=n_rows)
data['price'] = rng.normal(100, 20, size=n_rows)

# 42 interaction features (float64)
for i in range(21):
    data[f'cat_{i}_nan_wc'] = rng.normal(0, 1, size=n_rows)
    data[f'cat_{i}_wc'] = rng.normal(0, 1, size=n_rows)

# Misc float features
data['NaNs'] = rng.integers(0, 100, size=n_rows).astype(np.float32)
for i in range(7):
    data[f'extra_{i}'] = rng.normal(0, 1, size=n_rows)

# 19 digit features (int8)
for i in range(1, 10):
    data[f'digit{i}'] = rng.integers(-1, 10, size=n_rows).astype(np.int8)
for i in range(4):
    for j in range(i + 1, 5):
        data[f'digit_{i+1}_{j+1}'] = rng.integers(0, 121, size=n_rows).astype(np.int8)

# 210 pair features (int8) — C(21,2)
cats = [f'cat_{i}' for i in range(21)]
for i in range(len(cats)):
    for j in range(i + 1, len(cats)):
        data[f'{cats[i]}_{cats[j]}'] = rng.integers(0, 127, size=n_rows).astype(np.int8)

df = cudf.DataFrame(data)
del data
print(f"Shape: {df.shape}")
print(f"GPU memory: {df.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

## Feature Engineering: Round 1

Basic transformations on numeric columns.

In [ ]:
# Log and power transforms
df['price_log'] = cudf.Series(np.log1p(np.abs(df['price'].to_numpy())))
df['weight_sq'] = df['weight'] ** 2
df['price_per_weight'] = df['price'] / df['weight'].clip(lower=1)
df['weight_log'] = cudf.Series(np.log1p(np.abs(df['weight'].to_numpy())))
print(f"Round 1: {df.shape[1]} columns")

## Feature Engineering: Round 2

Categorical interaction features.

In [ ]:
# Categorical × numeric interactions
for i in range(7):
    df[f'cat_{i}_x_price'] = df[f'cat_{i}'] * df['price']
    df[f'cat_{i}_x_weight'] = df[f'cat_{i}'] * df['weight']
print(f"Round 2: {df.shape[1]} columns")

## Feature Engineering: Round 3

Aggregate statistics across digit features.

In [ ]:
# Digit aggregation
digit_cols = [f'digit{i}' for i in range(1, 10)]
df['digit_sum'] = sum(df[c].astype(np.int32) for c in digit_cols)
df['digit_max'] = cudf.concat([df[c].astype(np.int32) for c in digit_cols], axis=1).max(axis=1)
df['digit_min'] = cudf.concat([df[c].astype(np.int32) for c in digit_cols], axis=1).min(axis=1)
df['digit_range'] = df['digit_max'] - df['digit_min']
print(f"Round 3: {df.shape[1]} columns")

## Feature Engineering: Round 4

Squared categoricals and cross features.

In [ ]:
# Squared categoricals
for i in range(10):
    df[f'cat_{i}_sq'] = df[f'cat_{i}'] ** 2

# Extra ratio
for i in range(5):
    df[f'extra_{i}_ratio'] = df[f'extra_{i}'] / (df['price'].abs() + 1)
print(f"Round 4: {df.shape[1]} columns")

## Train/Test Split

In [ ]:
mask = rng.random(n_rows) < 0.8
train = df[mask]
test = df[~mask]
y_train = train['price']
y_test = test['price']
print(f"Train: {len(train):,} rows, Test: {len(test):,} rows")
print(f"Total GPU: ~{(df.memory_usage(deep=True).sum() + train.memory_usage(deep=True).sum() + test.memory_usage(deep=True).sum()) / 1024**2:.0f} MB")

## Modify Data (Triggers Change Detection)

FlowBook detects which columns changed using GPU-native comparison.

In [ ]:
# Normalize price and weight in-place
df['price'] = (df['price'] - df['price'].mean()) / df['price'].std()
df['weight'] = (df['weight'] - df['weight'].mean()) / df['weight'].std()
print("Normalized price and weight")

In [ ]:
%flowbook_status